# ML Training Pipeline -- Joint Molecular Optimization

**Notebook:** `02_ml_pipeline.ipynb`  
**Project:** Joint Molecular Optimization for Virtual Screening  
**Paper:** *Joint Molecular Optimization for Efficient Virtual Screening via Machine Learning*  
**Pipeline system:** STRATUM (Stratified Training and Ranking Architecture for Target-aware Unified Modeling)

---

## Overview

This notebook implements the four-pipeline ablation framework described in the paper. It trains classification and regression models to predict molecular activity and docking scores, while simultaneously evaluating ADMET desirability through a joint optimization scoring function.

### The Four-Pipeline Framework

| Pipeline | Multi-Target Data | Ligand Context | ADMET Features | Purpose |
|----------|:-----------------:|:--------------:|:--------------:|---|
| V1 | No | No | No | Fingerprint-only baseline |
| V2 | Yes | Yes | No | Multi-target + ligand conditioning |
| V3 | No | No | Yes | ADMET-augmented single target |
| V4 | Yes | Yes | Yes | Full joint optimization |

### Models Evaluated

**Classifiers** (active vs. decoy; metric: AUC-ROC): Random Forest, Naive Bayes, SVM (RBF), XGBoost, Neural Network.

**Regressors** (docking score prediction; metrics: MSE, R2): Random Forest, SVR, Linear Regression, XGBoost, Neural Network.

---

## Prerequisites

1. Complete `01_docking_pipeline.ipynb` to generate `docking_data.csv` for each target.
2. Mount Google Drive and verify CSV paths match the configuration in the final cell.
3. RDKit is assumed pre-installed in the runtime. Uncomment the install line in Cell 1 if needed.

## 1. Environment Setup, Imports, and Logging

Installs all required packages, imports libraries, creates the output directory tree, and initialises the `Tee` logger that mirrors all console output simultaneously to the notebook and a timestamped log file.

**Version notes:**
- `numpy==1.24.4` is pinned to avoid breaking RDKit C++ bindings in Colab.
- `tensorflow-cpu==2.16.1` is sufficient; GPU is not needed for these model sizes.
- `joblib` is available for optional model serialization (see Cell 10).

**Output directories created:**
```
Results_STRATUM/
    Images/          -- ROC curves, model comparison bar charts, CV line plots
    Full_Results/    -- Per-molecule prediction CSVs
    Rankings/        -- Top-k candidate lists and overlap tables
    Logs/            -- Timestamped console log
```

The `TRADEMARK_START` and `TRADEMARK_END` markers delimit each pipeline run in the log file for programmatic parsing.

In [ ]:
# If RDKit is not present in your runtime, uncomment the line below.
!pip install -q rdkit
!pip install -q scikit-learn pandas xgboost tensorflow-cpu==2.16.1 numpy==1.24.4 matplotlib seaborn joblib

import os, sys, datetime, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

# RDKit imports (assumes rdkit is available in the runtime)
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem, Descriptors, Crippen, Lipinski

# ML imports
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC, SVR
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import StratifiedKFold, GridSearchCV, train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import roc_auc_score, roc_curve, mean_squared_error, r2_score
import xgboost as xgb
import joblib
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

# ---------------------------
# RESULTS FOLDER + LOG (Tee)
# ---------------------------
RESULTS_ROOT = Path("/content/Results_STRATUM")
IMAGES_DIR = RESULTS_ROOT / "Images"
FULL_DIR   = RESULTS_ROOT / "Full_Results"
RANKS_DIR  = RESULTS_ROOT / "Rankings"
LOGS_DIR   = RESULTS_ROOT / "Logs"
for d in [IMAGES_DIR, FULL_DIR, RANKS_DIR, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Pipeline boundary markers
TRADEMARK_START = "=== STRATUM PIPELINE START ==="
TRADEMARK_END   = "=== STRATUM PIPELINE END   ==="

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
log_path = LOGS_DIR / f"stratum_log_{timestamp}.txt"

class Tee:
    """Write to both notebook (stdout) and a log file."""
    def __init__(self, *files):
        self.files = files
    def write(self, obj):
        for f in self.files:
            f.write(obj)
    def flush(self):
        for f in self.files:
            try:
                f.flush()
            except:
                pass

# open log file and set up Tee (prints go both to notebook and file)
_log_f = open(log_path, "w", encoding="utf-8")
tee = Tee(sys.stdout, _log_f)
sys.stdout = tee
print(TRADEMARK_START)
print(f"Log file: {log_path}\n")

## 2. Utility and Feature Engineering Functions

Defines the four core functions used by all pipeline versions to load data and build molecular feature matrices.

### `safe_read_csv(p)`
Loads a docking data CSV with file existence and column validation, raising informative errors early.

### `compute_admet_features(smiles_list)`
Computes six physicochemical descriptors per molecule using RDKit. Invalid SMILES receive NaN values replaced with column medians as a robust fallback.

| Descriptor | Abbreviation | Drug-likeness significance |
|---|---|---|
| Octanol-water partition coefficient | LogP | Lipophilicity; Lipinski limit |
| Topological polar surface area | TPSA | Membrane permeability predictor |
| Molecular weight | MolWt | Lipinski limit: <= 500 Da |
| H-bond donors | HDonors | Lipinski limit: <= 5 |
| H-bond acceptors | HAcceptors | Lipinski limit: <= 10 |
| Rotatable bonds | RotBonds | Conformational flexibility |

### `smiles_to_morgan(smiles_list)`
Encodes molecules as 2048-bit Morgan circular fingerprints at radius 2 (ECFP4). Returns float32 arrays for compatibility with sklearn and TensorFlow.

### `prepare_features(smiles_series, extra_smiles, include_admet)`
Assembles the final feature matrix for one pipeline version:
- If `extra_smiles` is provided (V2, V4), ligand SMILES are prefixed to molecule SMILES using the SMILES fragment separator `.` before fingerprinting, encoding binding-pocket context.
- If `include_admet` is True (V3, V4), six ADMET descriptors are concatenated to the fingerprint matrix.
- Returns the standardized feature matrix, the raw ADMET DataFrame, and the fitted scaler.

In [ ]:
# ---------------------------
# Utility & Feature functions
# ---------------------------
def safe_read_csv(p):
    p = Path(p)
    if not p.exists():
        raise FileNotFoundError(f"File not found: {p}")
    df = pd.read_csv(p)
    if 'SMILES' not in df.columns or 'docking_score' not in df.columns:
        raise ValueError(f"{p.name} must contain 'SMILES' and 'docking_score' columns")
    return df

def compute_admet_features(smiles_list):
    """Return DataFrame with LogP, TPSA, MolWt, HDonors, HAcceptors, RotBonds."""
    rows = []
    for s in smiles_list:
        try:
            m = Chem.MolFromSmiles(s) if isinstance(s, str) else None
        except:
            m = None
        if m:
            rows.append([
                float(Crippen.MolLogP(m)),
                float(Descriptors.TPSA(m)),
                float(Descriptors.MolWt(m)),
                int(Lipinski.NumHDonors(m)),
                int(Lipinski.NumHAcceptors(m)),
                int(Lipinski.NumRotatableBonds(m))
            ])
        else:
            rows.append([np.nan]*6)
    cols = ['LogP','TPSA','MolWt','HDonors','HAcceptors','RotBonds']
    df = pd.DataFrame(rows, columns=cols)
    # fallback: fillna with median for each column
    df = df.fillna(df.median())
    return df

def smiles_to_morgan(smiles_list, radius=2, n_bits=2048):
    fps = []
    for s in smiles_list:
        try:
            m = Chem.MolFromSmiles(s) if isinstance(s, str) else None
        except:
            m = None
        if m:
            bv = AllChem.GetMorganFingerprintAsBitVect(m, radius, nBits=n_bits)
            arr = np.zeros((n_bits,), dtype=np.int8)
            DataStructs.ConvertToNumpyArray(bv, arr)
            fps.append(arr)
        else:
            fps.append(np.zeros((n_bits,), dtype=np.int8))
    return np.array(fps, dtype=np.float32)

def prepare_features(smiles_series, extra_smiles=None, include_admet=False, n_bits=2048):
    """Return (X, admet_df, scaler). X standardized."""
    if extra_smiles is not None:
        combined = [f"{l}.{s}" for l,s in zip(extra_smiles, smiles_series)]
    else:
        combined = smiles_series.tolist()
    fps = smiles_to_morgan(combined, radius=2, n_bits=n_bits)
    admet_df = compute_admet_features(smiles_series)
    if include_admet:
        X = np.hstack([fps, admet_df.values])
    else:
        X = fps
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)
    return Xs, admet_df, scaler

## 3. Joint ADMET and Docking Score Combination

`compute_true_combined_score` produces the ground-truth multi-objective ranking criterion used to evaluate top-k compound retrieval. It implements the core joint optimization concept from the paper:

**Docking normalization** -- raw SMINA scores (kcal/mol, more negative = stronger binding) are inverted and min-max scaled to [0, 1] so stronger binders receive higher values.

**ADMET desirability** -- each of the six descriptors is mapped to [0, 1] where 1 is optimal (lower LogP deviation from 2.5, lower TPSA, lower MW, fewer donors/acceptors/rotatable bonds). Sub-scores are averaged into a single desirability value.

**Combined score** -- both normalized signals are jointly scaled via MinMaxScaler and combined at equal 50/50 weighting. A compound must score well on both binding affinity and ADMET to rank highly, which is the operationalization of joint optimization.

In [ ]:
# ---------------------------
# ADMET + combined scoring
# ---------------------------
def compute_true_combined_score(admet_df, docking_scores, weight_docking=0.5, weight_admet=0.5):
    """
    Creates a true_combined score where higher = better.
    docking_scores: lower is better (more negative better).
    We normalize docking to 0..1 (higher better) and compute normalized admet desirability,
    then combine.
    """
    ds = np.array(docking_scores, dtype=float)
    if np.ptp(ds) == 0:
        docking_norm = np.ones_like(ds)
    else:
        docking_norm = (ds.max() - ds) / (ds.max() - ds.min())  # higher = better
    # admet desirability
    logp = admet_df['LogP'].values; tpsa = admet_df['TPSA'].values; mw = admet_df['MolWt'].values
    hdon = admet_df['HDonors'].values; hacc = admet_df['HAcceptors'].values; rotb = admet_df['RotBonds'].values
    s_logp = 1 - np.abs(logp - 2.5)/5.0
    s_tpsa = 1 - (tpsa / (tpsa.max()+1e-9))
    s_mw = 1 - (mw / (mw.max()+1e-9))
    s_hdon = 1 - (hdon / (hdon.max()+1e-9))
    s_hacc = 1 - (hacc / (hacc.max()+1e-9))
    s_rot = 1 - (rotb / (rotb.max()+1e-9))
    # clip
    s_logp = np.clip(s_logp, 0, 1)
    s_tpsa = np.clip(s_tpsa, 0, 1)
    s_mw = np.clip(s_mw, 0, 1)
    s_hdon = np.clip(s_hdon, 0, 1)
    s_hacc = np.clip(s_hacc, 0, 1)
    s_rot = np.clip(s_rot, 0, 1)
    admet_score = (s_logp + s_tpsa + s_mw + s_hdon + s_hacc + s_rot) / 6.0
    # normalize docking_norm and admet_score to 0..1 using MinMax
    mm = MinMaxScaler()
    combined_norm = mm.fit_transform(np.vstack([docking_norm, admet_score]).T)
    docking_norm_n = combined_norm[:,0]
    admet_score_n = combined_norm[:,1]
    true_combined = weight_docking * docking_norm_n + weight_admet * admet_score_n
    return true_combined, admet_score

## 4. Neural Network Architectures

Two compact multi-layer perceptrons:

**Classifier:**
```
Input -> Dense(256, ReLU) -> Dropout(0.2) -> Dense(128, ReLU) -> Dense(1, sigmoid)
```
Outputs P(active) in [0, 1]. Loss: binary cross-entropy.

**Regressor:**
```
Input -> Dense(256, ReLU) -> Dropout(0.2) -> Dense(128, ReLU) -> Dense(1, linear)
```
Outputs a continuous docking score estimate. Loss: MSE.

Both use Adam optimizer. Dropout regularizes against the sparse, high-dimensional fingerprint input. These builder functions are called once per cross-validation fold to create fresh model instances, preventing weight carryover between folds. Early stopping (patience=3) is applied during fold training.

In [ ]:
# ---------------------------
# Simple NN builders (lightweight)
# ---------------------------
def build_classifier_nn(input_dim):
    model = models.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(128, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy')
    return model

def build_regressor_nn(input_dim):
    model = models.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(128, activation='relu'),
        layers.Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    return model

## 5. Visualization Functions

Three functions produce the diagnostic figures corresponding to Figures 1-4 in the paper. All output is saved as PNG files to `Images/`.

### `plot_overlapping_roc`
Overlapping ROC curves for all classifiers on one axis, with AUC annotated in each legend entry. Evaluated on the last cross-validation fold's held-out test set.

### `plot_model_comparison_bar`
Grouped bar chart of mean cross-validated metric per model, with standard deviation error bars and annotated values above each bar.

### `plot_cv_lines`
Per-fold metric trajectories for all models across the five CV folds. Reveals training stability: tight lines indicate consistent generalization; high variance indicates sensitivity to data partitioning.

In [ ]:
# ---------------------------
# Plotting helpers (research-grade)
# ---------------------------
def plot_overlapping_roc(models_clf, X_val, y_val, save_path, title_suffix=""):
    plt.figure(figsize=(8,6))
    for name, model in models_clf.items():
        try:
            if isinstance(model, tf.keras.Model) or 'keras' in str(type(model)).lower():
                probs = model.predict(X_val).flatten()
            else:
                probs = model.predict_proba(X_val)[:,1]
        except Exception as e:
            # fallback: some SVMs may not have predict_proba
            try:
                probs = model.decision_function(X_val)
                probs = (probs - probs.min())/(probs.max()-probs.min()+1e-9)
            except:
                continue
        fpr, tpr, _ = roc_curve(y_val, probs)
        auc_val = roc_auc_score(y_val, probs)
        plt.plot(fpr, tpr, lw=2, label=f"{name} (AUC={auc_val:.3f})")
    plt.plot([0,1],[0,1], color="black", lw=1, linestyle="--")
    plt.title(f"ROC Curves {title_suffix}")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.legend(loc="lower right", fontsize=9)
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

def plot_model_comparison_bar(metric_dict, metric_name, save_path):
    # metric_dict: {model_name: [scores across CV folds]}
    names = list(metric_dict.keys())
    means = [np.mean(metric_dict[n]) for n in names]
    stds  = [np.std(metric_dict[n]) for n in names]
    plt.figure(figsize=(10,6))
    # Matplotlib bar with errorbars
    x = np.arange(len(names))
    bars = plt.bar(x, means, yerr=stds, capsize=6)
    plt.xticks(x, names, rotation=30)
    plt.ylabel(metric_name)
    plt.title(f"Model comparison -- {metric_name}")
    # annotate bars with mean+/-std
    for i, (m,s) in enumerate(zip(means,stds)):
        plt.text(i, m + max(stds)*0.02, f"{m:.3f}+/-{s:.3f}", ha='center', va='bottom', fontsize=9)
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

def plot_cv_lines(metric_dict, metric_name, save_path):
    plt.figure(figsize=(10,6))
    for name, vals in metric_dict.items():
        plt.plot(range(1, len(vals)+1), vals, marker='o', lw=2, label=name)
    plt.xlabel("CV Fold")
    plt.ylabel(metric_name)
    plt.title(f"{metric_name} across CV folds")
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

## 6. Core Training Engine -- `evaluate_and_rank`

The central function executing training, evaluation, ranking, and output for one pipeline version.

**Step 1 -- Ground-truth ranking.** `compute_true_combined_score` produces reference top-k lists that model predictions are evaluated against.

**Step 2 -- Hyperparameter tuning.** Random Forest and XGBoost classifiers and regressors undergo `GridSearchCV` on `max_depth` (and `learning_rate` for XGBoost) with 3-fold inner CV.

**Step 3 -- 5-fold Stratified KFold CV.** All classifiers and regressors are trained per fold. The NN is re-instantiated fresh each fold to prevent weight carryover, and trained with early stopping.

**Step 4 -- Full-dataset predictions.** After CV, models from the final fold predict over the entire dataset for top-k overlap analysis.

**Step 5 -- Top-k overlap analysis.** For k in [10, 25, 50, 100], the model-selected top-k compounds are compared against the true top-k by `true_combined` score. Overlap percentage is the primary evaluation metric for the joint optimization goal.

**Step 6 -- Output files.** Saves per-version overlap tables, full prediction CSVs, top-k candidate CSVs, and all diagnostic plots.

In [ ]:
# ---------------------------
# Core: train, CV, evaluate and output
# ---------------------------
def evaluate_and_rank(X, admet_df, df_master, y_clf, y_reg, version_tag):
    n_mols = len(df_master)
    print(f"\n== Version {version_tag} dataset size: {n_mols} molecules ==")

    # compute true combined
    true_combined, admet_score = compute_true_combined_score(admet_df, y_reg, weight_docking=0.5, weight_admet=0.5)
    df_master = df_master.copy()
    df_master['ADMET_score'] = admet_score
    df_master['true_combined'] = true_combined

    # Stratified KFold for classification (use same indices for regression)
    n_splits = 5
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    # prepare models
    print("Preparing models and light tuning (RF/XGBoost)...")
    rf_clf = RandomForestClassifier(n_estimators=150, random_state=42)
    rf_clf = GridSearchCV(rf_clf, {'max_depth':[8,16]}, cv=3, scoring='roc_auc', n_jobs=1).fit(X, y_clf).best_estimator_

    xgb_clf = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', n_jobs=1)
    xgb_clf = GridSearchCV(xgb_clf, {'max_depth':[4,6], 'learning_rate':[0.05,0.1]}, cv=3, scoring='roc_auc', n_jobs=1).fit(X, y_clf).best_estimator_

    models_clf = {
        "RandomForest": rf_clf,
        "NaiveBayes": GaussianNB(),
        "SVM": SVC(probability=True, C=1.5, gamma='scale'),
        "XGBoost": xgb_clf,
        "NN": build_classifier_nn(X.shape[1])
    }

    rf_reg = RandomForestRegressor(n_estimators=150, random_state=42)
    rf_reg = GridSearchCV(rf_reg, {'max_depth':[8,16]}, cv=3, scoring='neg_mean_squared_error', n_jobs=1).fit(X, y_reg).best_estimator_
    xgb_reg = xgb.XGBRegressor(n_jobs=1)
    xgb_reg = GridSearchCV(xgb_reg, {'max_depth':[4,6], 'learning_rate':[0.05,0.1]}, cv=3, scoring='neg_mean_squared_error', n_jobs=1).fit(X, y_reg).best_estimator_

    models_reg = {
        "RandomForest": rf_reg,
        "SVR": SVR(C=2.0),
        "Linear": LinearRegression(),
        "XGBoost": xgb_reg,
        "NN": build_regressor_nn(X.shape[1])
    }

    # Containers for CV metrics per model
    auc_cv = {k:[] for k in models_clf}
    mse_cv = {k:[] for k in models_reg}
    r2_cv  = {k:[] for k in models_reg}

    fold = 0
    # We'll also keep last fold test X/y for plotting ROC overlap (use the last fold)
    last_X_test = None; last_y_test = None

    for train_idx, test_idx in skf.split(X, y_clf):
        fold += 1
        X_tr, X_te = X[train_idx], X[test_idx]
        ytr_cl, yte_cl = y_clf[train_idx], y_clf[test_idx]
        ytr_reg, yte_reg = y_reg[train_idx], y_reg[test_idx]

        # Train classifiers
        for name, model in models_clf.items():
            if name == "NN":
                # clone a fresh model each fold to avoid weight carryover
                model_fold = build_classifier_nn(X.shape[1])
                es = callbacks.EarlyStopping(monitor='loss', patience=3, restore_best_weights=True)
                model_fold.fit(X_tr, ytr_cl, epochs=20, batch_size=64, verbose=0, callbacks=[es])
                probs = model_fold.predict(X_te).flatten()
                # save fitted model for full-dataset predictions (keep the last fold's NN)
                models_clf[name] = model_fold
            else:
                model.fit(X_tr, ytr_cl)
                try:
                    probs = model.predict_proba(X_te)[:,1]
                except:
                    # fallback to decision_function->scaled probability
                    dec = model.decision_function(X_te)
                    probs = (dec - dec.min())/(dec.max()-dec.min()+1e-9)
            auc_cv[name].append(roc_auc_score(yte_cl, probs))

        # Train regressors
        for name, model in models_reg.items():
            if name == "NN":
                model_fold = build_regressor_nn(X.shape[1])
                es = callbacks.EarlyStopping(monitor='loss', patience=3, restore_best_weights=True)
                model_fold.fit(X_tr, ytr_reg, epochs=20, batch_size=64, verbose=0, callbacks=[es])
                preds = model_fold.predict(X_te).flatten()
                models_reg[name] = model_fold
            else:
                model.fit(X_tr, ytr_reg)
                preds = model.predict(X_te)
            mse_cv[name].append(mean_squared_error(yte_reg, preds))
            r2_cv[name].append(r2_score(yte_reg, preds))

        last_X_test = X_te; last_y_test = yte_cl

    # Print CV summary
    print("\n===== CROSS-VALIDATION SUMMARY (means +/- std) =====")
    for name in auc_cv:
        print(f"[Classifier CV] {name}: AUC = {np.mean(auc_cv[name]):.3f} +/- {np.std(auc_cv[name]):.3f}")
    for name in mse_cv:
        print(f"[Regressor CV] {name}: MSE = {np.mean(mse_cv[name]):.3f} +/- {np.std(mse_cv[name]):.3f}, R2 = {np.mean(r2_cv[name]):.3f} +/- {np.std(r2_cv[name]):.3f}")

    # Compute full-dataset predictions for ranking / overlaps
    print("\nComputing full-dataset predictions for ranking/overlap analysis...")
    X_full = X
    clf_probs_full = {}
    for name, model in models_clf.items():
        if isinstance(model, tf.keras.Model) or 'keras' in str(type(model)).lower():
            probs = model.predict(X_full).flatten()
        else:
            try:
                probs = model.predict_proba(X_full)[:,1]
            except:
                dec = model.decision_function(X_full)
                probs = (dec - dec.min())/(dec.max()-dec.min()+1e-9)
        clf_probs_full[name] = probs

    reg_preds_full = {}
    for name, model in models_reg.items():
        if isinstance(model, tf.keras.Model) or 'keras' in str(type(model)).lower():
            preds = model.predict(X_full).flatten()
        else:
            preds = model.predict(X_full)
        reg_preds_full[name] = preds

    # True top-k indices by true_combined
    ks = [10,25,50,100]
    true_topk = {k: np.argsort(df_master['true_combined'].values)[::-1][:k] for k in ks}

    overlap_records = []
    # For classifiers: rank by probability; for regressors: combine predicted docking with known ADMET
    for name, probs in clf_probs_full.items():
        for k in ks:
            topk_idx = np.argsort(probs)[::-1][:k]
            overlap = len(set(topk_idx) & set(true_topk[k]))
            pct = overlap / k * 100.0
            overlap_records.append({'version': version_tag, 'model': name, 'type': 'classifier', 'k':k, 'overlap':overlap, 'pct':pct})

    # regressors: create predicted combined (pred docking -> higher is better)
    for name, preds in reg_preds_full.items():
        ds = np.array(preds, dtype=float)
        if np.ptp(ds) == 0:
            docking_norm = np.ones_like(ds)
        else:
            docking_norm = (ds.max() - ds) / (ds.max() - ds.min())
        mm = MinMaxScaler()
        comb_norm = mm.fit_transform(np.vstack([docking_norm, admet_score]).T)
        pred_combined = 0.5*comb_norm[:,0] + 0.5*comb_norm[:,1]
        for k in ks:
            topk_idx = np.argsort(pred_combined)[::-1][:k]
            overlap = len(set(topk_idx) & set(true_topk[k]))
            pct = overlap / k * 100.0
            overlap_records.append({'version': version_tag, 'model': name, 'type': 'regressor', 'k':k, 'overlap':overlap, 'pct':pct})

    df_overlaps = pd.DataFrame(overlap_records)
    df_overlaps.to_csv(RANKS_DIR / f"overlaps_{version_tag}_{timestamp}.csv", index=False)
    print(f"Saved overlap table: {RANKS_DIR / f'overlaps_{version_tag}_{timestamp}.csv'}")

    # Save full dataset with predictions
    df_out = df_master.copy()
    for name, arr in clf_probs_full.items(): df_out[f'prob_{name}'] = arr
    for name, arr in reg_preds_full.items(): df_out[f'pred_{name}'] = arr
    df_out.to_csv(FULL_DIR / f"full_predictions_{version_tag}_{timestamp}.csv", index=False)
    print(f"Saved full predictions: {FULL_DIR / f'full_predictions_{version_tag}_{timestamp}.csv'}")

    # Save top-k CSVs
    for rec in overlap_records:
        mod, typ, k = rec['model'], rec['type'], rec['k']
        if typ == 'classifier':
            scores = clf_probs_full[mod]
        else:
            preds = reg_preds_full[mod]
            ds = np.array(preds, dtype=float)
            if np.ptp(ds) == 0:
                docking_norm = np.ones_like(ds)
            else:
                docking_norm = (ds.max() - ds) / (ds.max() - ds.min())
            mm = MinMaxScaler()
            comb_norm = mm.fit_transform(np.vstack([docking_norm, admet_score]).T)
            scores = 0.5*comb_norm[:,0] + 0.5*comb_norm[:,1]
        topk_idx = np.argsort(scores)[::-1][:k]
        df_out.iloc[topk_idx].to_csv(RANKS_DIR / f"top_{k}_{mod}_{typ}_{version_tag}_{timestamp}.csv", index=False)

    # Plots: use last fold testset for ROC overlap
    roc_path = IMAGES_DIR / f"ROC_overlap_{version_tag}_{timestamp}.png"
    plot_overlapping_roc(models_clf, last_X_test, last_y_test, roc_path, title_suffix=f"(version {version_tag})")
    print(f"Saved ROC overlap plot: {roc_path}")

    # Model comparison bar (AUC)
    bar_path = IMAGES_DIR / f"Model_comp_AUC_{version_tag}_{timestamp}.png"
    plot_model_comparison_bar(auc_cv, "AUC", bar_path)
    print(f"Saved model comparison bar: {bar_path}")

    # CV lines
    cv_auc_path = IMAGES_DIR / f"CV_AUC_{version_tag}_{timestamp}.png"
    plot_cv_lines(auc_cv, "AUC", cv_auc_path)
    print(f"Saved CV AUC lines: {cv_auc_path}")

    cv_r2_path = IMAGES_DIR / f"CV_R2_{version_tag}_{timestamp}.png"
    plot_cv_lines(r2_cv, "R2", cv_r2_path)
    print(f"Saved CV R2 lines: {cv_r2_path}")

    cv_mse_path = IMAGES_DIR / f"CV_MSE_{version_tag}_{timestamp}.png"
    plot_cv_lines(mse_cv, "MSE", cv_mse_path)
    print(f"Saved CV MSE lines: {cv_mse_path}")

    # Top-k summary print
    print("\nTop-k Overlap Summary (first 20 rows):")
    if not df_overlaps.empty:
        print(df_overlaps.head(20).to_string(index=False))
    else:
        print("No overlaps recorded.")

    return df_overlaps, df_out

## 7. Pipeline Runner -- `run_version`

Orchestrates data loading, label assignment, feature preparation, and calls `evaluate_and_rank` for one pipeline version.

**Label assignment** is inferred from the CSV filename: files containing `active` receive label 1, `decoy` receives label 0. This convention is enforced by `01_docking_pipeline.ipynb`.

**Ligand SMILES placeholder:** For V2 and V4, aspirin SMILES (`CC(=O)Oc1ccccc1C(=O)O`) is used as a structural placeholder for ligand-conditioned fingerprinting. In production, replace this with the actual co-crystallized ligand SMILES for each target from the DUD-E `.mol2` files.

**Docking score coercion:** Non-numeric entries in the docking_score column (from SMINA parsing failures) are replaced with the column median before training.

In [ ]:
# ---------------------------
# Pipeline runner wrapper
# ---------------------------
def run_version(files_list, include_ligand=False, include_admet=False, version_tag="V"):
    print("\n" + "="*60)
    print(f"RUNNING VERSION: {version_tag}")
    print("="*60)
    # load CSVs
    dfs = []
    for p in files_list:
        df = safe_read_csv(p)
        name = Path(p).name.lower()
        if 'active' in name:
            df['label'] = 1
        elif 'decoy' in name:
            df['label'] = 0
        else:
            raise ValueError("Input filenames must indicate 'active' or 'decoy'")
        if include_ligand:
            df['Ligand_SMILES'] = 'CC(=O)Oc1ccccc1C(=O)O'  # placeholder
        dfs.append(df)
    df_all = pd.concat(dfs, ignore_index=True)
    print(f"Loaded {len(df_all)} molecules for {version_tag}")

    X, admet_df, scaler = prepare_features(df_all['SMILES'], extra_smiles=(df_all['Ligand_SMILES'] if include_ligand else None), include_admet=include_admet)
    y_clf = df_all['label'].values
    y_reg = pd.to_numeric(df_all['docking_score'], errors='coerce').fillna(df_all['docking_score'].median()).values

    overlaps_df, full_preds_df = evaluate_and_rank(X, admet_df, df_all, y_clf, y_reg, version_tag)
    print(f"VERSION {version_tag} COMPLETE")
    return overlaps_df, full_preds_df

## 8. Run All Four Pipeline Versions

Executes the complete four-version ablation study sequentially. Update `MACHINE_DATA` to match your Google Drive directory before running.

| Version | Files | include_ligand | include_admet |
|---|---|:---:|:---:|
| V1 | parp1 only | No | No |
| V2 | all 4 targets | Yes | No |
| V3 | parp1 only | No | Yes |
| V4 | all 4 targets | Yes | Yes |

A combined overlap table across all four versions is saved to `Rankings/all_overlaps_<timestamp>.csv` on completion. All console output is mirrored to `Logs/stratum_log_<timestamp>.txt`.

In [ ]:
# ---------------------------
# Run the four versions
# ---------------------------
FILES_VER1 = [
    '/content/drive/MyDrive/MACHINE DATA/ACTIVES/docking_data_active-parp1.csv',
    '/content/drive/MyDrive/MACHINE DATA/DECOYS/docking_data_decoy-parp1.csv'
]
FILES_VER2 = [
    '/content/drive/MyDrive/MACHINE DATA/ACTIVES/docking_data_active-parp1.csv',
    '/content/drive/MyDrive/MACHINE DATA/DECOYS/docking_data_decoy-parp1.csv',
    '/content/drive/MyDrive/MACHINE DATA/ACTIVES/docking_data_actives-ital.csv',
    '/content/drive/MyDrive/MACHINE DATA/DECOYS/docking_data_decoy-ital.csv',
    '/content/drive/MyDrive/MACHINE DATA/ACTIVES/docking_data_actives-mapk2.csv',
    '/content/drive/MyDrive/MACHINE DATA/DECOYS/docking_data_decoy-mapk2.csv',
    '/content/drive/MyDrive/MACHINE DATA/ACTIVES/docking_data_actives-tgfr1.csv',
    '/content/drive/MyDrive/MACHINE DATA/DECOYS/docking_data_decoy-tgfr1.csv'
]

results_agg = []
try:
    r1 = run_version(FILES_VER1, include_ligand=False, include_admet=False, version_tag="V1_noLig_noADMET")
    results_agg.append(("V1", r1))
    r2 = run_version(FILES_VER2, include_ligand=True, include_admet=False, version_tag="V2_multiLig_noADMET")
    results_agg.append(("V2", r2))
    r3 = run_version(FILES_VER1, include_ligand=False, include_admet=True, version_tag="V3_noLig_withADMET")
    results_agg.append(("V3", r3))
    r4 = run_version(FILES_VER2, include_ligand=True, include_admet=True, version_tag="V4_multiLig_withADMET")
    results_agg.append(("V4", r4))
    print("\nALL VERSIONS COMPLETE")
except Exception as e:
    print("ERROR during pipeline:", e)
    import traceback; traceback.print_exc()

# summary and finalize
print("\nSaving combined overlap summaries...")
all_overlaps = []
for tag, (ov, full) in results_agg:
    if ov is not None:
        all_overlaps.append(ov)
if all_overlaps:
    combined = pd.concat(all_overlaps, ignore_index=True)
    combined.to_csv(RANKS_DIR / f"all_overlaps_{timestamp}.csv", index=False)
    print(f"Combined overlaps saved: {RANKS_DIR / f'all_overlaps_{timestamp}.csv'}")

print("\n" + TRADEMARK_END)
# restore stdout and close log file
sys.stdout.flush()
sys.stdout = sys.__stdout__
_log_f.close()

print(f"Pipeline finished. Results directory: {RESULTS_ROOT}")
print(f"Images: {IMAGES_DIR}")
print(f"Full CSVs: {FULL_DIR}")
print(f"Rankings: {RANKS_DIR}")
print(f"Log file: {log_path}")

## 9. (Optional) Download Trained Models

The code above does not serialize models to disk by default. Run this cell after the pipeline completes to save all trained sklearn and XGBoost models with joblib, Keras neural networks in TensorFlow SavedModel format, and download everything as a zip archive.

This cell is independent of the pipeline -- it reads `models_clf` and `models_reg` from the last completed `evaluate_and_rank` call. If you want models from a specific version, re-run that version in isolation and then run this cell immediately after.

**To reload saved models:**
```python
import joblib, tensorflow as tf
rf = joblib.load('Models/clf_RandomForest.joblib')
nn = tf.keras.models.load_model('Models/clf_NN')
```

In [ ]:
import zipfile
from google.colab import files
from pathlib import Path

MODELS_DIR = RESULTS_ROOT / "Models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Save sklearn / XGBoost classifiers
for name, model in models_clf.items():
    if isinstance(model, tf.keras.Model) or 'keras' in str(type(model)).lower():
        model.save(str(MODELS_DIR / f"clf_{name}"))
    else:
        joblib.dump(model, MODELS_DIR / f"clf_{name}.joblib")

# Save sklearn / XGBoost regressors
for name, model in models_reg.items():
    if isinstance(model, tf.keras.Model) or 'keras' in str(type(model)).lower():
        model.save(str(MODELS_DIR / f"reg_{name}"))
    else:
        joblib.dump(model, MODELS_DIR / f"reg_{name}.joblib")

print(f"Models saved to: {MODELS_DIR}")

# Zip and download
zip_path = "/content/STRATUM_models.zip"
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fp in MODELS_DIR.rglob('*'):
        if fp.is_file():
            zf.write(fp, fp.relative_to(RESULTS_ROOT))

print(f"Archive: {zip_path}")
files.download(zip_path)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# STRATUM Pipeline -- Single-Cell Execution

**Pipeline system:** STRATUM (Stratified Training and Ranking Architecture for Target-aware Unified Modeling)  
**Paper:** *Joint Molecular Optimization for Efficient Virtual Screening via Machine Learning* -- Naman Jain

---

This cell contains the complete STRATUM machine learning pipeline in a single executable block.
It is provided for convenience when a sequential multi-cell notebook execution is not preferred,
or when running the pipeline as a scheduled or automated job.

**What this cell does, in order:**

1. Installs all required packages.
2. Imports all libraries.
3. Creates the output directory structure under `/content/Results_STRATUM/`.
4. Initialises dual-stream logging (notebook console and timestamped log file).
5. Defines all utility, feature engineering, scoring, neural network, and visualization functions.
6. Runs all four pipeline versions sequentially:
   - V1: Morgan fingerprint baseline, single target (PARP1), no ADMET descriptors.
   - V2: Ligand-conditioned fingerprints, four targets (PARP1, ITAL, MAPK2, TGFR1), no ADMET descriptors.
   - V3: Morgan fingerprints augmented with ADMET descriptors, single target, no ligand conditioning.
   - V4: Full joint optimization -- ligand-conditioned fingerprints, four targets, ADMET descriptors.
7. For each version: tunes RF and XGBoost hyperparameters via GridSearchCV, runs 5-fold stratified
   cross-validation across all five classifiers and five regressors, computes top-k overlap metrics
   against the ground-truth joint (docking + ADMET) ranking, and saves ROC curves, CV line plots,
   model comparison bar charts, full prediction CSVs, and top-k candidate lists.
8. Saves a combined overlap table across all four versions.
9. Restores stdout and closes the log file.

**Before running, update the following paths to match your Google Drive structure:**

```
FILES_VER1  -- paths to PARP1 active and decoy docking CSVs
FILES_VER2  -- paths to all four target active and decoy docking CSVs
```

All outputs are saved to `/content/Results_STRATUM/`. To download models after the run completes,
execute the optional download block provided at the end of `02_ml_pipeline.ipynb`.

**Note on ligand SMILES:** Pipelines V2 and V4 use a fixed reference SMILES
(`CC(=O)Oc1ccccc1C(=O)O`) as a structural placeholder for ligand-conditioned fingerprinting
rather than the per-target co-crystallized ligand from the DUD-E crystal_ligand.mol2 files.
See the README for a full discussion of this design choice and guidance for researchers
extending this work with target-specific crystal ligands.

---



In [1]:
# -------------------------------
# INSTALL REQUIRED PACKAGES
# -------------------------------
# If RDKit not present in your runtime, uncomment the rdkit install line below
# (Note: installing rdkit in Colab may take a while.)
# !pip install -q rdkit
!pip install -q scikit-learn pandas xgboost tensorflow-cpu==2.16.1 numpy==1.24.4 matplotlib seaborn joblib

import os, sys, datetime, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

# RDKit imports (assumes rdkit is available in the runtime)
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem, Descriptors, Crippen, Lipinski

# ML imports
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC, SVR
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import StratifiedKFold, GridSearchCV, train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import roc_auc_score, roc_curve, mean_squared_error, r2_score
import xgboost as xgb
import joblib
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

# ---------------------------
# RESULTS FOLDER + LOG (Tee)
# ---------------------------
RESULTS_ROOT = Path("/content/Results_STRATUM")
IMAGES_DIR = RESULTS_ROOT / "Images"
FULL_DIR   = RESULTS_ROOT / "Full_Results"
RANKS_DIR  = RESULTS_ROOT / "Rankings"
LOGS_DIR   = RESULTS_ROOT / "Logs"
for d in [IMAGES_DIR, FULL_DIR, RANKS_DIR, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# trademark markers
TRADEMARK_START = "=== AI PIPELINE START ==="
TRADEMARK_END   = "===  AI PIPELINE END   ==="

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
log_path = LOGS_DIR / f"STRATUM_{timestamp}.txt"

class Tee:
    """Write to both notebook (stdout) and a log file."""
    def __init__(self, *files):
        self.files = files
    def write(self, obj):
        for f in self.files:
            f.write(obj)
    def flush(self):
        for f in self.files:
            try:
                f.flush()
            except:
                pass

# open log file and set up Tee (prints go both to notebook and file)
_log_f = open(log_path, "w", encoding="utf-8")
tee = Tee(sys.stdout, _log_f)
sys.stdout = tee
print(TRADEMARK_START)
print(f"Log file: {log_path}\n")

# ---------------------------
# Utility & Feature functions
# ---------------------------
def safe_read_csv(p):
    p = Path(p)
    if not p.exists():
        raise FileNotFoundError(f"File not found: {p}")
    df = pd.read_csv(p)
    if 'SMILES' not in df.columns or 'docking_score' not in df.columns:
        raise ValueError(f"{p.name} must contain 'SMILES' and 'docking_score' columns")
    return df

def compute_admet_features(smiles_list):
    """Return DataFrame with LogP, TPSA, MolWt, HDonors, HAcceptors, RotBonds."""
    rows = []
    for s in smiles_list:
        try:
            m = Chem.MolFromSmiles(s) if isinstance(s, str) else None
        except:
            m = None
        if m:
            rows.append([
                float(Crippen.MolLogP(m)),
                float(Descriptors.TPSA(m)),
                float(Descriptors.MolWt(m)),
                int(Lipinski.NumHDonors(m)),
                int(Lipinski.NumHAcceptors(m)),
                int(Lipinski.NumRotatableBonds(m))
            ])
        else:
            rows.append([np.nan]*6)
    cols = ['LogP','TPSA','MolWt','HDonors','HAcceptors','RotBonds']
    df = pd.DataFrame(rows, columns=cols)
    # fallback: fillna with median for each column
    df = df.fillna(df.median())
    return df

def smiles_to_morgan(smiles_list, radius=2, n_bits=2048):
    fps = []
    for s in smiles_list:
        try:
            m = Chem.MolFromSmiles(s) if isinstance(s, str) else None
        except:
            m = None
        if m:
            bv = AllChem.GetMorganFingerprintAsBitVect(m, radius, nBits=n_bits)
            arr = np.zeros((n_bits,), dtype=np.int8)
            DataStructs.ConvertToNumpyArray(bv, arr)
            fps.append(arr)
        else:
            fps.append(np.zeros((n_bits,), dtype=np.int8))
    return np.array(fps, dtype=np.float32)

def prepare_features(smiles_series, extra_smiles=None, include_admet=False, n_bits=2048):
    """Return (X, admet_df, scaler). X standardized."""
    if extra_smiles is not None:
        combined = [f"{l}.{s}" for l,s in zip(extra_smiles, smiles_series)]
    else:
        combined = smiles_series.tolist()
    fps = smiles_to_morgan(combined, radius=2, n_bits=n_bits)
    admet_df = compute_admet_features(smiles_series)
    if include_admet:
        X = np.hstack([fps, admet_df.values])
    else:
        X = fps
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)
    return Xs, admet_df, scaler

# ---------------------------
# ADMET + combined scoring
# ---------------------------
def compute_true_combined_score(admet_df, docking_scores, weight_docking=0.5, weight_admet=0.5):
    """
    Creates a true_combined score where higher = better.
    docking_scores: lower is better (more negative better).
    We normalize docking to 0..1 (higher better) and compute normalized admet desirability,
    then combine.
    """
    ds = np.array(docking_scores, dtype=float)
    if np.ptp(ds) == 0:
        docking_norm = np.ones_like(ds)
    else:
        docking_norm = (ds.max() - ds) / (ds.max() - ds.min())  # higher = better
    # admet desirability
    logp = admet_df['LogP'].values; tpsa = admet_df['TPSA'].values; mw = admet_df['MolWt'].values
    hdon = admet_df['HDonors'].values; hacc = admet_df['HAcceptors'].values; rotb = admet_df['RotBonds'].values
    s_logp = 1 - np.abs(logp - 2.5)/5.0
    s_tpsa = 1 - (tpsa / (tpsa.max()+1e-9))
    s_mw = 1 - (mw / (mw.max()+1e-9))
    s_hdon = 1 - (hdon / (hdon.max()+1e-9))
    s_hacc = 1 - (hacc / (hacc.max()+1e-9))
    s_rot = 1 - (rotb / (rotb.max()+1e-9))
    # clip
    s_logp = np.clip(s_logp, 0, 1)
    s_tpsa = np.clip(s_tpsa, 0, 1)
    s_mw = np.clip(s_mw, 0, 1)
    s_hdon = np.clip(s_hdon, 0, 1)
    s_hacc = np.clip(s_hacc, 0, 1)
    s_rot = np.clip(s_rot, 0, 1)
    admet_score = (s_logp + s_tpsa + s_mw + s_hdon + s_hacc + s_rot) / 6.0
    # normalize docking_norm and admet_score to 0..1 using MinMax
    mm = MinMaxScaler()
    combined_norm = mm.fit_transform(np.vstack([docking_norm, admet_score]).T)
    docking_norm_n = combined_norm[:,0]
    admet_score_n = combined_norm[:,1]
    true_combined = weight_docking * docking_norm_n + weight_admet * admet_score_n
    return true_combined, admet_score

# ---------------------------
# Simple NN builders (lightweight)
# ---------------------------
def build_classifier_nn(input_dim):
    model = models.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(128, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy')
    return model

def build_regressor_nn(input_dim):
    model = models.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(128, activation='relu'),
        layers.Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    return model

# ---------------------------
# Plotting helpers (research-grade)
# ---------------------------
def plot_overlapping_roc(models_clf, X_val, y_val, save_path, title_suffix=""):
    plt.figure(figsize=(8,6))
    for name, model in models_clf.items():
        try:
            if isinstance(model, tf.keras.Model) or 'keras' in str(type(model)).lower():
                probs = model.predict(X_val).flatten()
            else:
                probs = model.predict_proba(X_val)[:,1]
        except Exception as e:
            # fallback: some SVMs may not have predict_proba
            try:
                probs = model.decision_function(X_val)
                probs = (probs - probs.min())/(probs.max()-probs.min()+1e-9)
            except:
                continue
        fpr, tpr, _ = roc_curve(y_val, probs)
        auc_val = roc_auc_score(y_val, probs)
        plt.plot(fpr, tpr, lw=2, label=f"{name} (AUC={auc_val:.3f})")
    plt.plot([0,1],[0,1], color="black", lw=1, linestyle="--")
    plt.title(f"ROC Curves {title_suffix}")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.legend(loc="lower right", fontsize=9)
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

def plot_model_comparison_bar(metric_dict, metric_name, save_path):
    # metric_dict: {model_name: [scores across CV folds]}
    names = list(metric_dict.keys())
    means = [np.mean(metric_dict[n]) for n in names]
    stds  = [np.std(metric_dict[n]) for n in names]
    plt.figure(figsize=(10,6))
    # Matplotlib bar with errorbars
    x = np.arange(len(names))
    bars = plt.bar(x, means, yerr=stds, capsize=6)
    plt.xticks(x, names, rotation=30)
    plt.ylabel(metric_name)
    plt.title(f"Model comparison — {metric_name}")
    # annotate bars with mean±std
    for i, (m,s) in enumerate(zip(means,stds)):
        plt.text(i, m + max(stds)*0.02, f"{m:.3f}±{s:.3f}", ha='center', va='bottom', fontsize=9)
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

def plot_cv_lines(metric_dict, metric_name, save_path):
    plt.figure(figsize=(10,6))
    for name, vals in metric_dict.items():
        plt.plot(range(1, len(vals)+1), vals, marker='o', lw=2, label=name)
    plt.xlabel("CV Fold")
    plt.ylabel(metric_name)
    plt.title(f"{metric_name} across CV folds")
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

# ---------------------------
# Core: train, CV, evaluate and output
# ---------------------------
def evaluate_and_rank(X, admet_df, df_master, y_clf, y_reg, version_tag):
    n_mols = len(df_master)
    print(f"\n== Version {version_tag} dataset size: {n_mols} molecules ==")

    # compute true combined
    true_combined, admet_score = compute_true_combined_score(admet_df, y_reg, weight_docking=0.5, weight_admet=0.5)
    df_master = df_master.copy()
    df_master['ADMET_score'] = admet_score
    df_master['true_combined'] = true_combined

    # Stratified KFold for classification (use same indices for regression)
    n_splits = 5
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    # prepare models
    print("Preparing models and light tuning (RF/XGBoost)...")
    rf_clf = RandomForestClassifier(n_estimators=150, random_state=42)
    rf_clf = GridSearchCV(rf_clf, {'max_depth':[8,16]}, cv=3, scoring='roc_auc', n_jobs=1).fit(X, y_clf).best_estimator_

    xgb_clf = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', n_jobs=1)
    xgb_clf = GridSearchCV(xgb_clf, {'max_depth':[4,6], 'learning_rate':[0.05,0.1]}, cv=3, scoring='roc_auc', n_jobs=1).fit(X, y_clf).best_estimator_

    models_clf = {
        "RandomForest": rf_clf,
        "NaiveBayes": GaussianNB(),
        "SVM": SVC(probability=True, C=1.5, gamma='scale'),
        "XGBoost": xgb_clf,
        "NN": build_classifier_nn(X.shape[1])
    }

    rf_reg = RandomForestRegressor(n_estimators=150, random_state=42)
    rf_reg = GridSearchCV(rf_reg, {'max_depth':[8,16]}, cv=3, scoring='neg_mean_squared_error', n_jobs=1).fit(X, y_reg).best_estimator_
    xgb_reg = xgb.XGBRegressor(n_jobs=1)
    xgb_reg = GridSearchCV(xgb_reg, {'max_depth':[4,6], 'learning_rate':[0.05,0.1]}, cv=3, scoring='neg_mean_squared_error', n_jobs=1).fit(X, y_reg).best_estimator_

    models_reg = {
        "RandomForest": rf_reg,
        "SVR": SVR(C=2.0),
        "Linear": LinearRegression(),
        "XGBoost": xgb_reg,
        "NN": build_regressor_nn(X.shape[1])
    }

    # Containers for CV metrics per model
    auc_cv = {k:[] for k in models_clf}
    mse_cv = {k:[] for k in models_reg}
    r2_cv  = {k:[] for k in models_reg}

    fold = 0
    # We'll also keep last fold test X/y for plotting ROC overlap (use the last fold)
    last_X_test = None; last_y_test = None

    for train_idx, test_idx in skf.split(X, y_clf):
        fold += 1
        X_tr, X_te = X[train_idx], X[test_idx]
        ytr_cl, yte_cl = y_clf[train_idx], y_clf[test_idx]
        ytr_reg, yte_reg = y_reg[train_idx], y_reg[test_idx]

        # Train classifiers
        for name, model in models_clf.items():
            if name == "NN":
                # clone a fresh model each fold to avoid weight carryover
                model_fold = build_classifier_nn(X.shape[1])
                es = callbacks.EarlyStopping(monitor='loss', patience=3, restore_best_weights=True)
                model_fold.fit(X_tr, ytr_cl, epochs=20, batch_size=64, verbose=0, callbacks=[es])
                probs = model_fold.predict(X_te).flatten()
                # save fitted model for full-dataset predictions (keep the last fold's NN)
                models_clf[name] = model_fold
            else:
                model.fit(X_tr, ytr_cl)
                try:
                    probs = model.predict_proba(X_te)[:,1]
                except:
                    # fallback to decision_function->scaled probability
                    dec = model.decision_function(X_te)
                    probs = (dec - dec.min())/(dec.max()-dec.min()+1e-9)
            auc_cv[name].append(roc_auc_score(yte_cl, probs))

        # Train regressors
        for name, model in models_reg.items():
            if name == "NN":
                model_fold = build_regressor_nn(X.shape[1])
                es = callbacks.EarlyStopping(monitor='loss', patience=3, restore_best_weights=True)
                model_fold.fit(X_tr, ytr_reg, epochs=20, batch_size=64, verbose=0, callbacks=[es])
                preds = model_fold.predict(X_te).flatten()
                models_reg[name] = model_fold
            else:
                model.fit(X_tr, ytr_reg)
                preds = model.predict(X_te)
            mse_cv[name].append(mean_squared_error(yte_reg, preds))
            r2_cv[name].append(r2_score(yte_reg, preds))

        last_X_test = X_te; last_y_test = yte_cl

    # Print CV summary
    print("\n===== CROSS-VALIDATION SUMMARY (means ± std) =====")
    for name in auc_cv:
        print(f"[Classifier CV] {name}: AUC = {np.mean(auc_cv[name]):.3f} ± {np.std(auc_cv[name]):.3f}")
    for name in mse_cv:
        print(f"[Regressor CV] {name}: MSE = {np.mean(mse_cv[name]):.3f} ± {np.std(mse_cv[name]):.3f}, R2 = {np.mean(r2_cv[name]):.3f} ± {np.std(r2_cv[name]):.3f}")

    # Compute full-dataset predictions for ranking / overlaps
    print("\nComputing full-dataset predictions for ranking/overlap analysis...")
    X_full = X
    clf_probs_full = {}
    for name, model in models_clf.items():
        if isinstance(model, tf.keras.Model) or 'keras' in str(type(model)).lower():
            probs = model.predict(X_full).flatten()
        else:
            try:
                probs = model.predict_proba(X_full)[:,1]
            except:
                dec = model.decision_function(X_full)
                probs = (dec - dec.min())/(dec.max()-dec.min()+1e-9)
        clf_probs_full[name] = probs

    reg_preds_full = {}
    for name, model in models_reg.items():
        if isinstance(model, tf.keras.Model) or 'keras' in str(type(model)).lower():
            preds = model.predict(X_full).flatten()
        else:
            preds = model.predict(X_full)
        reg_preds_full[name] = preds

    # True top-k indices by true_combined
    ks = [10,25,50,100]
    true_topk = {k: np.argsort(df_master['true_combined'].values)[::-1][:k] for k in ks}

    overlap_records = []
    # For classifiers: rank by probability; for regressors: combine predicted docking with known ADMET
    for name, probs in clf_probs_full.items():
        for k in ks:
            topk_idx = np.argsort(probs)[::-1][:k]
            overlap = len(set(topk_idx) & set(true_topk[k]))
            pct = overlap / k * 100.0
            overlap_records.append({'version': version_tag, 'model': name, 'type': 'classifier', 'k':k, 'overlap':overlap, 'pct':pct})

    # regressors: create predicted combined (pred docking -> higher is better)
    for name, preds in reg_preds_full.items():
        ds = np.array(preds, dtype=float)
        if np.ptp(ds) == 0:
            docking_norm = np.ones_like(ds)
        else:
            docking_norm = (ds.max() - ds) / (ds.max() - ds.min())
        mm = MinMaxScaler()
        comb_norm = mm.fit_transform(np.vstack([docking_norm, admet_score]).T)
        pred_combined = 0.5*comb_norm[:,0] + 0.5*comb_norm[:,1]
        for k in ks:
            topk_idx = np.argsort(pred_combined)[::-1][:k]
            overlap = len(set(topk_idx) & set(true_topk[k]))
            pct = overlap / k * 100.0
            overlap_records.append({'version': version_tag, 'model': name, 'type': 'regressor', 'k':k, 'overlap':overlap, 'pct':pct})

    df_overlaps = pd.DataFrame(overlap_records)
    df_overlaps.to_csv(RANKS_DIR / f"overlaps_{version_tag}_{timestamp}.csv", index=False)
    print(f"Saved overlap table: {RANKS_DIR / f'overlaps_{version_tag}_{timestamp}.csv'}")

    # Save full dataset with predictions
    df_out = df_master.copy()
    for name, arr in clf_probs_full.items(): df_out[f'prob_{name}'] = arr
    for name, arr in reg_preds_full.items(): df_out[f'pred_{name}'] = arr
    df_out.to_csv(FULL_DIR / f"full_predictions_{version_tag}_{timestamp}.csv", index=False)
    print(f"Saved full predictions: {FULL_DIR / f'full_predictions_{version_tag}_{timestamp}.csv'}")

    # Save top-k CSVs
    for rec in overlap_records:
        mod, typ, k = rec['model'], rec['type'], rec['k']
        if typ == 'classifier':
            scores = clf_probs_full[mod]
        else:
            preds = reg_preds_full[mod]
            ds = np.array(preds, dtype=float)
            if np.ptp(ds) == 0:
                docking_norm = np.ones_like(ds)
            else:
                docking_norm = (ds.max() - ds) / (ds.max() - ds.min())
            mm = MinMaxScaler()
            comb_norm = mm.fit_transform(np.vstack([docking_norm, admet_score]).T)
            scores = 0.5*comb_norm[:,0] + 0.5*comb_norm[:,1]
        topk_idx = np.argsort(scores)[::-1][:k]
        df_out.iloc[topk_idx].to_csv(RANKS_DIR / f"top_{k}_{mod}_{typ}_{version_tag}_{timestamp}.csv", index=False)

    # Plots: use last fold testset for ROC overlap
    roc_path = IMAGES_DIR / f"ROC_overlap_{version_tag}_{timestamp}.png"
    plot_overlapping_roc(models_clf, last_X_test, last_y_test, roc_path, title_suffix=f"(version {version_tag})")
    print(f"Saved ROC overlap plot: {roc_path}")

    # Model comparison bar (AUC)
    bar_path = IMAGES_DIR / f"Model_comp_AUC_{version_tag}_{timestamp}.png"
    plot_model_comparison_bar(auc_cv, "AUC", bar_path)
    print(f"Saved model comparison bar: {bar_path}")

    # CV lines
    cv_auc_path = IMAGES_DIR / f"CV_AUC_{version_tag}_{timestamp}.png"
    plot_cv_lines(auc_cv, "AUC", cv_auc_path)
    print(f"Saved CV AUC lines: {cv_auc_path}")

    cv_r2_path = IMAGES_DIR / f"CV_R2_{version_tag}_{timestamp}.png"
    plot_cv_lines(r2_cv, "R2", cv_r2_path)
    print(f"Saved CV R2 lines: {cv_r2_path}")

    cv_mse_path = IMAGES_DIR / f"CV_MSE_{version_tag}_{timestamp}.png"
    plot_cv_lines(mse_cv, "MSE", cv_mse_path)
    print(f"Saved CV MSE lines: {cv_mse_path}")

    # Top-k summary print
    print("\nTop-k Overlap Summary (first 20 rows):")
    if not df_overlaps.empty:
        print(df_overlaps.head(20).to_string(index=False))
    else:
        print("No overlaps recorded.")

    return df_overlaps, df_out

# ---------------------------
# Pipeline runner wrapper
# ---------------------------
def run_version(files_list, include_ligand=False, include_admet=False, version_tag="V"):
    print("\n" + "="*60)
    print(f"RUNNING VERSION: {version_tag}")
    print("="*60)
    # load CSVs
    dfs = []
    for p in files_list:
        df = safe_read_csv(p)
        name = Path(p).name.lower()
        if 'active' in name:
            df['label'] = 1
        elif 'decoy' in name:
            df['label'] = 0
        else:
            raise ValueError("Input filenames must indicate 'active' or 'decoy'")
        if include_ligand:
            df['Ligand_SMILES'] = 'CC(=O)Oc1ccccc1C(=O)O'
        dfs.append(df)
    df_all = pd.concat(dfs, ignore_index=True)
    print(f"Loaded {len(df_all)} molecules for {version_tag}")

    X, admet_df, scaler = prepare_features(df_all['SMILES'], extra_smiles=(df_all['Ligand_SMILES'] if include_ligand else None), include_admet=include_admet)
    y_clf = df_all['label'].values
    y_reg = pd.to_numeric(df_all['docking_score'], errors='coerce').fillna(df_all['docking_score'].median()).values

    overlaps_df, full_preds_df = evaluate_and_rank(X, admet_df, df_all, y_clf, y_reg, version_tag)
    print(f"VERSION {version_tag} COMPLETE ✅")
    return overlaps_df, full_preds_df

# ---------------------------
# Run the four versions
# ---------------------------
FILES_VER1 = [
    '/content/drive/MyDrive/MACHINE DATA/ACTIVES/docking_data_active-parp1.csv',
    '/content/drive/MyDrive/MACHINE DATA/DECOYS/docking_data_decoy-parp1.csv'
]
FILES_VER2 = [
    '/content/drive/MyDrive/MACHINE DATA/ACTIVES/docking_data_active-parp1.csv',
    '/content/drive/MyDrive/MACHINE DATA/DECOYS/docking_data_decoy-parp1.csv',
    '/content/drive/MyDrive/MACHINE DATA/ACTIVES/docking_data_actives-ital.csv',
    '/content/drive/MyDrive/MACHINE DATA/DECOYS/docking_data_decoy-ital.csv',
    '/content/drive/MyDrive/MACHINE DATA/ACTIVES/docking_data_actives-mapk2.csv',
    '/content/drive/MyDrive/MACHINE DATA/DECOYS/docking_data_decoy-mapk2.csv',
    '/content/drive/MyDrive/MACHINE DATA/ACTIVES/docking_data_actives-tgfr1.csv',
    '/content/drive/MyDrive/MACHINE DATA/DECOYS/docking_data_decoy-tgfr1.csv'
]

results_agg = []
try:
    r1 = run_version(FILES_VER1, include_ligand=False, include_admet=False, version_tag="V1_noLig_noADMET")
    results_agg.append(("V1", r1))
    r2 = run_version(FILES_VER2, include_ligand=True, include_admet=False, version_tag="V2_multiLig_noADMET")
    results_agg.append(("V2", r2))
    r3 = run_version(FILES_VER1, include_ligand=False, include_admet=True, version_tag="V3_noLig_withADMET")
    results_agg.append(("V3", r3))
    r4 = run_version(FILES_VER2, include_ligand=True, include_admet=True, version_tag="V4_multiLig_withADMET")
    results_agg.append(("V4", r4))
    print("\nALL VERSIONS COMPLETE ✅")
except Exception as e:
    print("ERROR during pipeline:", e)
    import traceback; traceback.print_exc()

# summary and finalize
print("\nSaving combined overlap summaries...")
all_overlaps = []
for tag, (ov, full) in results_agg:
    if ov is not None:
        all_overlaps.append(ov)
if all_overlaps:
    combined = pd.concat(all_overlaps, ignore_index=True)
    combined.to_csv(RANKS_DIR / f"all_overlaps_{timestamp}.csv", index=False)
    print(f"Combined overlaps saved: {RANKS_DIR / f'all_overlaps_{timestamp}.csv'}")

print("\n" + TRADEMARK_END)
# restore stdout and close log file
sys.stdout.flush()
sys.stdout = sys.__stdout__
_log_f.close()

print(f"Pipeline finished. Results directory: {RESULTS_ROOT}")
print(f"Images: {IMAGES_DIR}")
print(f"Full CSVs: {FULL_DIR}")
print(f"Rankings: {RANKS_DIR}")
print(f"Log file: {log_path}")



77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
Saved overlap table: /content/Results_PhoenixAI2/Rankings/overlaps_V4_multiLig_withADMET_20260517_012723.csv
Saved full predictions: /content/Results_PhoenixAI2/Full_Results/full_predictions_V4_multiLig_withADMET_20260517_012723.csv
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
Saved ROC overlap plot: /content/Results_PhoenixAI2/Images/ROC_overlap_V4_multiLig_withADMET_20260517_012723.png
Saved model comparison bar: /content/Results_PhoenixAI2/Images/Model_comp_AUC_V4_multiLig_withADMET_20260517_012723.png
Saved CV AUC lines: /content/Results_PhoenixAI2/Images/CV_AUC_V4_multiLig_withADMET_20260517_012723.png
Saved CV R2 lines: /content/Results_PhoenixAI2/Images/CV_R2_V4_multiLig_withADMET_20260517_012723.png
Saved CV MSE lines: /content/Results_PhoenixAI2/Images/CV_MSE_V4_multiLig_withADMET_20260517_012723.png

Top-k Overlap Summary (first 20 rows):
              version        model       type   k  overlap  pct
V4_